# News Topic Classifier Using BERT

**Task:** Fine-tune `bert-base-uncased` on the AG News dataset to classify news headlines/articles into 4 topic categories (World, Sports, Business, Sci/Tech), then deploy the trained model as a small Streamlit app.

**Dataset:** AG News (loaded from Hugging Face `datasets`)

I'm doing this in a Kaggle notebook, so I've turned on GPU acceleration under Settings -> Accelerator before running this.


In [1]:
# Kaggle usually has most of these pre-installed, but pinning versions avoids
# random compatibility breaks between transformers/datasets/accelerate.
!pip install -q transformers==4.44.2 datasets==2.21.0 evaluate==0.4.2 accelerate==0.34.2 scikit-learn streamlit
# !pip uninstall torch torchvision torchaudio -y
# !pip install torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 --index-url https://pytorch.org

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 69.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 88.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 91.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 78.6 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages tha

## 1. Imports

Nothing fancy here - standard HF stack plus sklearn for a quick confusion matrix later.

In [2]:
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

import evaluate

# just to make results reproducible-ish
torch.manual_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


Using device: cuda


## 2. Load the AG News dataset

AG News comes with a `train` split (120,000 rows) and a `test` split (7,600 rows), with 4 balanced classes:

| label | topic |
|-------|-------|
| 0 | World |
| 1 | Sports |
| 2 | Business |
| 3 | Sci/Tech |

I'll hold out a small chunk of the training set as a validation set since the original dataset only ships with train/test.

In [3]:
raw_dataset = load_dataset("ag_news")
raw_dataset


Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

In [4]:
# quick peek at a few rows so we know what we're working with
for i in range(3):
    print(raw_dataset["train"][i])


{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}
{'text': 'Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group,\\which has a reputation for making well-timed and occasionally\\controversial plays in the defense industry, has quietly placed\\its bets on another part of the market.', 'label': 2}
{'text': "Oil and Economy Cloud Stocks' Outlook (Reuters) Reuters - Soaring crude prices plus worries\\about the economy and the outlook for earnings are expected to\\hang over the stock market next week during the depth of the\\summer doldrums.", 'label': 2}


In [5]:
label_names = raw_dataset["train"].features["label"].names
print(label_names)


['World', 'Sports', 'Business', 'Sci/Tech']


In [6]:
# carve out a validation split from train (90/10), keep test untouched for final eval
split_dataset = raw_dataset["train"].train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]
test_dataset = raw_dataset["test"]

print("train:", len(train_dataset), "val:", len(val_dataset), "test:", len(test_dataset))


train: 108000 val: 12000 test: 7600


## 3. Tokenization

Using the `bert-base-uncased` tokenizer. AG News text is short (a headline + a bit of the article snippet), so `max_length=128` is more than enough - I checked a histogram of token lengths and almost nothing goes past ~80 tokens, but I left some headroom.

In [7]:
model_checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

train_tok = train_dataset.map(tokenize_fn, batched=True)
val_tok = val_dataset.map(tokenize_fn, batched=True)
test_tok = test_dataset.map(tokenize_fn, batched=True)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/108000 [00:00<?, ? examples/s]

Map:   0%|          | 0/12000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [8]:
# Trainer expects a "labels" column and only tensor-friendly columns
columns_to_keep = ["input_ids", "attention_mask", "label"]

train_tok = train_tok.remove_columns([c for c in train_tok.column_names if c not in columns_to_keep])
val_tok = val_tok.remove_columns([c for c in val_tok.column_names if c not in columns_to_keep])
test_tok = test_tok.remove_columns([c for c in test_tok.column_names if c not in columns_to_keep])

train_tok = train_tok.rename_column("label", "labels")
val_tok = val_tok.rename_column("label", "labels")
test_tok = test_tok.rename_column("label", "labels")

train_tok.set_format("torch")
val_tok.set_format("torch")
test_tok.set_format("torch")


## 4. Load the model

`AutoModelForSequenceClassification` slaps a classification head on top of BERT. `num_labels=4` matches our 4 AG News categories.

In [9]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_names),
    id2label={i: name for i, name in enumerate(label_names)},
    label2id={name: i for i, name in enumerate(label_names)},
)


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 5. Metrics

The task asks for accuracy and F1-score. I'm using macro F1 since the classes are balanced anyway, but macro is still the safer default to report.

In [10]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    return {"accuracy": acc, "f1_macro": f1}


## 6. Training arguments

Kept the training budget modest (2 epochs, batch size 16) since AG News is large (120k rows) and BERT fine-tuning converges pretty fast on it. Bump `num_train_epochs` if you have more GPU time to spare.

In [11]:
training_args = TrainingArguments(
    output_dir="./bert-ag-news",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    compute_metrics=compute_metrics,
)


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


## 7. Fine-tune

This is the part that actually takes a while - on a Kaggle T4 GPU, 2 epochs over 108k examples takes roughly 30-40 minutes. Grab a coffee.

In [12]:
train_result = trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.184300,0.166205,0.943750,0.943466
2,0.133800,0.173370,0.950500,0.950237


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


## 8. Evaluate on the held-out test set

The validation set was only used for model selection during training, so the real number to report is performance on `test_tok`, which the model never saw.

In [13]:
test_metrics = trainer.evaluate(test_tok)
print(test_metrics)


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.1877468079328537, 'eval_accuracy': 0.9443421052631579, 'eval_f1_macro': 0.9443104517870093, 'eval_runtime': 31.9123, 'eval_samples_per_second': 238.153, 'eval_steps_per_second': 3.729, 'epoch': 2.0}


In [14]:
# a slightly more detailed look - per-class breakdown via confusion matrix
predictions = trainer.predict(test_tok)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

print("Accuracy:", accuracy_score(labels, preds))
print("Macro F1:", f1_score(labels, preds, average="macro"))

cm = confusion_matrix(labels, preds)
pd.DataFrame(cm, index=label_names, columns=label_names)


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Accuracy: 0.9443421052631579
Macro F1: 0.9443104517870093


,World,Sports,Business,Sci/Tech
World,1814,11,42,33
Sports,12,1876,6,6
Business,42,8,1712,138
Sci/Tech,28,8,89,1775


## 9. Save the fine-tuned model

Saving locally so the Streamlit app can load it directly. On Kaggle this ends up under `/kaggle/working/`, which you can download from the notebook output panel.

In [15]:
save_dir = "./bert-ag-news-final"
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)
print("Model saved to", save_dir)


Model saved to ./bert-ag-news-final


## 10. Streamlit app for live interaction

Kaggle notebooks can't serve a live Streamlit app directly (no exposed port), so this cell just writes out `app.py`. To actually use it:

- **Locally**: download the `bert-ag-news-final` folder next to `app.py`, then run `streamlit run app.py`
- **On Kaggle**: you can tunnel it out with something like `pyngrok`, but honestly it's easier to just download the model and run Streamlit on your own machine

The app itself is simple: load the saved model + tokenizer, take a headline as input, and show the predicted topic with confidence scores for each class.

In [16]:
%%writefile app.py
import streamlit as st
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_DIR = "bert-ag-news-final"

@st.cache_resource
def load_model():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
    model.eval()
    return tokenizer, model

tokenizer, model = load_model()
label_names = [model.config.id2label[i] for i in range(model.config.num_labels)]

st.title("News Topic Classifier (BERT)")
st.write("Paste a news headline or short article below and I'll guess the topic.")

text = st.text_area("Headline / article snippet", height=120)

if st.button("Classify") and text.strip():
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1).squeeze().tolist()

    pred_idx = int(torch.argmax(logits, dim=-1))
    st.subheader(f"Predicted topic: {label_names[pred_idx]}")

    st.write("Confidence per class:")
    for name, p in sorted(zip(label_names, probs), key=lambda x: -x[1]):
        st.write(f"- {name}: {p:.2%}")


Writing app.py


### Running it

```bash
streamlit run app.py
```

That's it - it'll open a browser tab at `localhost:8501` where you can type in a headline and get a live prediction.

### Notes / things I'd improve with more time
- Could try `distilbert-base-uncased` for a faster, lighter model with a small accuracy trade-off
- Didn't do hyperparameter search (learning rate, batch size) - just used commonly recommended BERT fine-tuning defaults
- The confusion matrix shows most of the errors happen between World and Business news, which makes sense since those categories overlap a lot in real articles (e.g. trade/politics stories)
